In [1]:
#ATM Cash Replenishment Optimization Engine

#Objective:
#Use machine learning predictions and business rules to calculate recommended ATM refill amounts, safety buffers, and risk levels.

In [2]:
import pandas as pd

df = pd.read_csv("atm_cash_management_dataset.csv")

In [3]:
df["Calculated_Cash"] = (
    df["Previous_Day_Cash_Level"]
    - df["Total_Withdrawals"]
    + df["Total_Deposits"]
)

df["Current_Cash_Level"] = df["Calculated_Cash"].clip(lower=0)

In [4]:
df[["Calculated_Cash", "Current_Cash_Level"]].head()

,Calculated_Cash,Current_Cash_Level
0,64811,64811
1,60399,60399
2,60486,60486
3,47115,47115
4,95871,95871


In [5]:
(df["Calculated_Cash"] < 0).sum()

np.int64(46)

In [6]:
df["Cash_Shortage"] = df["Calculated_Cash"].apply(
    lambda x: abs(x) if x < 0 else 0
)

In [7]:
df[["Calculated_Cash", "Cash_Shortage"]].head()

,Calculated_Cash,Cash_Shortage
0,64811,0
1,60399,0
2,60486,0
3,47115,0
4,95871,0


In [8]:
df["Cash_Shortage"].describe()

count     5658.000000
mean        93.927006
std       1378.206332
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max      49433.000000
Name: Cash_Shortage, dtype: float64

In [9]:
import numpy as np

conditions = [
    (df["Holiday_Flag"] == 1) & (df["Special_Event_Flag"] == 1),
    (df["Holiday_Flag"] == 1) & (df["Special_Event_Flag"] == 0),
    (df["Holiday_Flag"] == 0) & (df["Special_Event_Flag"] == 1)
]

choices = [0.35, 0.25, 0.30]

df["Safety_Buffer_Percentage"] = np.select(
    conditions,
    choices,
    default=0.15
)

In [10]:
df["Safety_Buffer_Percentage"].value_counts()

Safety_Buffer_Percentage
0.15    4997
0.30     536
0.25     115
0.35      10
Name: count, dtype: int64

In [11]:
import joblib

model = joblib.load("linear_regression_model.pkl")

In [12]:
import pandas as pd
import joblib

df_encoded = pd.read_csv("atm_encoded_dataset.csv")

model = joblib.load("linear_regression_model.pkl")

In [13]:
y = df_encoded["Cash_Demand_Next_Day"]

X = df_encoded.drop(
    columns=[
        "ATM_ID",
        "Date",
        "Cash_Demand_Next_Day"
    ]
)

In [14]:
df_encoded["Predicted_Demand"] = model.predict(X)

In [15]:
df_encoded["Predicted_Demand"].head()

0    50164.038540
1    58832.292581
2    33149.150165
3    42611.634633
4    34180.759963
Name: Predicted_Demand, dtype: float64

In [16]:
df_encoded["Safety_Buffer_Percentage"] = df["Safety_Buffer_Percentage"]
df_encoded["Current_Cash_Level"] = df["Current_Cash_Level"]
df_encoded["Cash_Shortage"] = df["Cash_Shortage"]

In [17]:
df_encoded["Safety_Buffer_Amount"] = (
    df_encoded["Predicted_Demand"]
    * df_encoded["Safety_Buffer_Percentage"]
)

In [18]:
print(df.shape)
print(df_encoded.shape)


(5658, 17)
(5658, 34)


In [19]:
df_encoded[
    [
        "Predicted_Demand",
        "Safety_Buffer_Percentage",
        "Current_Cash_Level"
    ]
].head()

,Predicted_Demand,Safety_Buffer_Percentage,Current_Cash_Level
0,50164.038540,0.15,64811
1,58832.292581,0.15,60399
2,33149.150165,0.15,60486
3,42611.634633,0.15,47115
4,34180.759963,0.15,95871


In [20]:
df_encoded["Required_Cash"] = (
    df_encoded["Predicted_Demand"]
    + df_encoded["Safety_Buffer_Amount"]
)

In [21]:
df_encoded["Current_Cash_Level"]

0       64811
1       60399
2       60486
3       47115
4       95871
        ...  
5653    67563
5654    73761
5655    82787
5656    68236
5657    38316
Name: Current_Cash_Level, Length: 5658, dtype: int64

In [22]:
df_encoded["Recommended_Refill"] = (
    df_encoded["Required_Cash"]
    - df_encoded["Current_Cash_Level"]
)

In [23]:
df_encoded["Recommended_Refill"] = (
    df_encoded["Recommended_Refill"]
    .clip(lower=0)
)

In [24]:
df_encoded[
    [
        "Predicted_Demand",
        "Current_Cash_Level",
        "Required_Cash",
        "Recommended_Refill"
    ]
].head()

,Predicted_Demand,Current_Cash_Level,Required_Cash,Recommended_Refill
0,50164.038540,64811,57688.644321,0.000000
1,58832.292581,60399,67657.136468,7258.136468
2,33149.150165,60486,38121.522689,0.000000
3,42611.634633,47115,49003.379828,1888.379828
4,34180.759963,95871,39307.873957,0.000000


In [25]:
import numpy as np

conditions = [
    df_encoded["Recommended_Refill"] >= 50000,
    df_encoded["Recommended_Refill"] >= 20000
]

choices = ["High Risk", "Medium Risk"]

df_encoded["Risk_Level"] = np.select(
    conditions,
    choices,
    default="Low Risk"
)

In [26]:
df_encoded[
    [
        "Predicted_Demand",
        "Current_Cash_Level",
        "Safety_Buffer_Amount",
        "Required_Cash",
        "Recommended_Refill",
        "Risk_Level"
    ]
].head(10)

,Predicted_Demand,Current_Cash_Level,Safety_Buffer_Amount,Required_Cash,Recommended_Refill,Risk_Level
0,50164.038540,64811,7524.605781,57688.644321,0.000000,Low Risk
1,58832.292581,60399,8824.843887,67657.136468,7258.136468,Low Risk
2,33149.150165,60486,4972.372525,38121.522689,0.000000,Low Risk
3,42611.634633,47115,6391.745195,49003.379828,1888.379828,Low Risk
4,34180.759963,95871,5127.113994,39307.873957,0.000000,Low Risk
5,46386.839726,62883,11596.709932,57983.549658,0.000000,Low Risk
6,34140.419720,55516,5121.062958,39261.482678,0.000000,Low Risk
7,68250.727220,11911,10237.609083,78488.336303,66577.336303,High Risk
8,21268.344155,95676,3190.251623,24458.595778,0.000000,Low Risk
9,45208.863342,50828,6781.329501,51990.192843,1162.192843,Low Risk


In [27]:
cols = [
    "Predicted_Demand",
    "Safety_Buffer_Amount",
    "Required_Cash",
    "Recommended_Refill"
]

df_encoded[cols] = df_encoded[cols].round(0)

In [28]:
print("Total Refill Amount:",
      df_encoded["Recommended_Refill"].sum())
print(
    (df_encoded["Risk_Level"] == "High Risk").sum()
)
print(
    (df_encoded["Risk_Level"] == "Medium Risk").sum()
)
print(
    (df_encoded["Risk_Level"] == "Low Risk").sum()
)

Total Refill Amount: 62440714.0
363
881
4414


In [29]:
df_encoded["Priority_Rank"] = (
    df_encoded["Recommended_Refill"]
    .rank(ascending=False)
)
df_encoded[
    [
        "Recommended_Refill",
        "Risk_Level",
        "Priority_Rank"
    ]
].sort_values(
    by="Priority_Rank"
).head(10)

,Recommended_Refill,Risk_Level,Priority_Rank
1125,133470.0,High Risk,1.0
694,112652.0,High Risk,2.0
1789,107398.0,High Risk,3.0
2293,100450.0,High Risk,4.0
4030,99206.0,High Risk,5.0
2613,98004.0,High Risk,6.0
1688,97600.0,High Risk,7.0
1710,95512.0,High Risk,8.0
251,95350.0,High Risk,9.0
4117,95135.0,High Risk,10.0


In [30]:
df_encoded["Risk_Level"].value_counts()

Risk_Level
Low Risk       4414
Medium Risk     881
High Risk       363
Name: count, dtype: int64

In [31]:
df_encoded.groupby("Risk_Level")["Recommended_Refill"].sum()

Risk_Level
High Risk      23777505.0
Low Risk        9725341.0
Medium Risk    28937868.0
Name: Recommended_Refill, dtype: float64

In [32]:
df_encoded.to_csv(
    "optimizer_results.csv",
    index=False
)

In [33]:
print(df.columns)

Index(['ATM_ID', 'Date', 'Day_of_Week', 'Time_of_Day', 'Total_Withdrawals',
       'Total_Deposits', 'Location_Type', 'Holiday_Flag', 'Special_Event_Flag',
       'Previous_Day_Cash_Level', 'Weather_Condition',
       'Nearby_Competitor_ATMs', 'Cash_Demand_Next_Day', 'Calculated_Cash',
       'Current_Cash_Level', 'Cash_Shortage', 'Safety_Buffer_Percentage'],
      dtype='object')


In [34]:
print(df.columns.tolist())

['ATM_ID', 'Date', 'Day_of_Week', 'Time_of_Day', 'Total_Withdrawals', 'Total_Deposits', 'Location_Type', 'Holiday_Flag', 'Special_Event_Flag', 'Previous_Day_Cash_Level', 'Weather_Condition', 'Nearby_Competitor_ATMs', 'Cash_Demand_Next_Day', 'Calculated_Cash', 'Current_Cash_Level', 'Cash_Shortage', 'Safety_Buffer_Percentage']


In [35]:
df_encoded.to_csv("optimizer_results.csv", index=False)

In [36]:
df_encoded.to_csv("optimizer_results.csv", index=False)

In [37]:
dashboard_df = df_encoded[
    [
        "ATM_ID",
        "Predicted_Demand",
        "Current_Cash_Level",
        "Required_Cash",
        "Recommended_Refill",
        "Risk_Level",
        "Priority_Rank"
    ]
]

In [38]:
dashboard_df.to_csv(
    "dashboard_data.csv",
    index=False
)

In [39]:
df_encoded.columns

Index(['ATM_ID', 'Date', 'Total_Withdrawals', 'Total_Deposits', 'Holiday_Flag',
       'Special_Event_Flag', 'Previous_Day_Cash_Level',
       'Nearby_Competitor_ATMs', 'Cash_Demand_Next_Day', 'Year', 'Month',
       'Day', 'Quarter', 'Day_of_Week_Monday', 'Day_of_Week_Saturday',
       'Day_of_Week_Sunday', 'Day_of_Week_Thursday', 'Day_of_Week_Tuesday',
       'Day_of_Week_Wednesday', 'Time_of_Day_Evening', 'Time_of_Day_Morning',
       'Time_of_Day_Night', 'Location_Type_Gas Station', 'Location_Type_Mall',
       'Location_Type_Standalone', 'Location_Type_Supermarket',
       'Weather_Condition_Cloudy', 'Weather_Condition_Rainy',
       'Weather_Condition_Snowy', 'Predicted_Demand',
       'Safety_Buffer_Percentage', 'Current_Cash_Level', 'Cash_Shortage',
       'Safety_Buffer_Amount', 'Required_Cash', 'Recommended_Refill',
       'Risk_Level', 'Priority_Rank'],
      dtype='object')

In [40]:
dashboard_df.head()

,ATM_ID,Predicted_Demand,Current_Cash_Level,Required_Cash,Recommended_Refill,Risk_Level,Priority_Rank
0,ATM_0041,50164.0,64811,57689.0,0.0,Low Risk,3952.5
1,ATM_0007,58832.0,60399,67657.0,7258.0,Low Risk,1870.0
2,ATM_0014,33149.0,60486,38122.0,0.0,Low Risk,3952.5
3,ATM_0029,42612.0,47115,49003.0,1888.0,Low Risk,2135.5
4,ATM_0028,34181.0,95871,39308.0,0.0,Low Risk,3952.5


In [41]:
dashboard_df.to_csv(
    "dashboard_data.csv",
    index=False
)

In [42]:
print(df_encoded.columns)

Index(['ATM_ID', 'Date', 'Total_Withdrawals', 'Total_Deposits', 'Holiday_Flag',
       'Special_Event_Flag', 'Previous_Day_Cash_Level',
       'Nearby_Competitor_ATMs', 'Cash_Demand_Next_Day', 'Year', 'Month',
       'Day', 'Quarter', 'Day_of_Week_Monday', 'Day_of_Week_Saturday',
       'Day_of_Week_Sunday', 'Day_of_Week_Thursday', 'Day_of_Week_Tuesday',
       'Day_of_Week_Wednesday', 'Time_of_Day_Evening', 'Time_of_Day_Morning',
       'Time_of_Day_Night', 'Location_Type_Gas Station', 'Location_Type_Mall',
       'Location_Type_Standalone', 'Location_Type_Supermarket',
       'Weather_Condition_Cloudy', 'Weather_Condition_Rainy',
       'Weather_Condition_Snowy', 'Predicted_Demand',
       'Safety_Buffer_Percentage', 'Current_Cash_Level', 'Cash_Shortage',
       'Safety_Buffer_Amount', 'Required_Cash', 'Recommended_Refill',
       'Risk_Level', 'Priority_Rank'],
      dtype='object')
